In [ ]:
df = pd.read_csv('Superstore.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

9994

Total Business Performance (Core KPI)

In [ ]:
query = """
SELECT
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    SUM(Profit) / SUM(Sales) AS profit_margin,
    COUNT(DISTINCT `Order ID`) AS total_orders
FROM sales;
"""

pd.read_sql(query, conn)

,total_sales,total_profit,profit_margin,total_orders
0,2.297201e+06,286397.0217,0.124672,5009


Sales Trend Over Time

In [ ]:
query = """
SELECT
    strftime('%Y-%m', `Order Date`) AS month,
    SUM(Sales) AS total_sales
FROM sales
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, conn)

,month,total_sales
0,None,2.297201e+06


Top Products (Revenue Drivers)

In [ ]:
query = """
SELECT
    `Product Name`,
    SUM(Sales) AS total_sales
FROM sales
GROUP BY `Product Name`
ORDER BY total_sales DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,Product Name,total_sales
0,Canon imageCLASS 2200 Advanced Copier,61599.824
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384
2,Cisco TelePresence System EX90 Videoconferenci...,22638.480
3,HON 5400 Series Task Chairs for Big and Tall,21870.576
4,GBC DocuBind TL300 Electric Binding System,19823.479
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.500
6,Hewlett Packard LaserJet 3310 Copier,18839.686
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.895
8,GBC DocuBind P400 Electric Binding System,17965.068
9,High Speed Automatic Electric Letter Opener,17030.312


Category Performance

In [ ]:
query = """
SELECT
    Category,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit
FROM sales
GROUP BY Category
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,Category,total_sales,total_profit
0,Technology,836154.0330,145454.9481
1,Furniture,741999.7953,18451.2728
2,Office Supplies,719047.0320,122490.8008


Customer Segmentation

In [ ]:
query = """
SELECT
    `Customer Name`,
    SUM(Sales) AS total_spent
FROM sales
GROUP BY `Customer Name`
ORDER BY total_spent DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,Customer Name,total_spent
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


Regional Analysis

In [ ]:
query = """
SELECT
    Region,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit
FROM sales
GROUP BY Region
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,Region,total_sales,total_profit
0,West,725457.8245,108418.4489
1,East,678781.2400,91522.7800
2,Central,501239.8908,39706.3625
3,South,391721.9050,46749.4303


Discount Impact

In [ ]:
query = """
SELECT
    Discount,
    AVG(Profit) AS avg_profit
FROM sales
GROUP BY Discount
ORDER BY Discount;
"""

pd.read_sql(query, conn)

,Discount,avg_profit
0,0.00,66.900292
1,0.10,96.055074
2,0.15,27.288298
3,0.20,24.702572
4,0.30,-45.679636
5,0.32,-88.560656
6,0.40,-111.927429
7,0.45,-226.646464
8,0.50,-310.703456
9,0.60,-43.077212


Data Cleaning + Feature Engineering

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df.dtypes

,0
Row ID,int64
Order ID,object
Order Date,datetime64[ns]
Ship Date,datetime64[ns]
Ship Mode,object
Customer ID,object
Customer Name,object
Segment,object
Country,object
City,object


In [ ]:
df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [ ]:
df = df.dropna()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['Profit Margin'] = df['Profit'] / df['Sales']

In [ ]:
df['Order Year'] = df['Order Date'].dt.year

In [ ]:
df['Order Month'] = df['Order Date'].dt.month

In [ ]:
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')

In [ ]:
def discount_category(x):
    if x == 0:
        return 'No Discount'
    elif x < 0.2:
        return 'Low Discount'
    else:
        return 'High Discount'

df['Discount Category'] = df['Discount'].apply(discount_category)

In [ ]:
df.to_csv('cleaned_sales_data.csv', index=False)